# Task 3 — Query Optimization: Partition Pruning

## Overview

This notebook tests partition pruning as an optimization. The taxi_trips Delta table is physically partitioned by pickup_year and pickup_month (for example, pickup_year=2024/pickup_month=1/...). Partition pruning lets Spark skip entire partition folders at planning time, instead of reading every partition and filtering rows afterward.

## Query design

None of the six queries in the Task 1/2 library filter by pickup_year or pickup_month. They are all full history aggregations, so there is no partition for them to skip regardless of filter.

To test pruning, we use a simplified version of Query 1's measure: trip count per pickup zone, for a single month (January 2024), on taxi_trips alone, without the taxi_zones join from the original query. We also tested a version with the join; see the Discussion section for what that showed.

Two versions of the same query are compared:

- Baseline: filters on pickup_datetime, which is not a partition column.
- Optimized: filters directly on pickup_year and pickup_month, the actual partition columns.

Both versions select the same data, so any difference in time or plan comes from the filter column, not from a different result.

## Methodology

Each version runs one discarded warm-up query, then five timed repetitions. We report the median, since Spark run times are right skewed by occasional GC pauses.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from data_intensive_computing.benchmark_helpers import (
    build_spark_session,
    register_delta_tables,
    timed_run,
    capture_explain,
    verify_same_results,
    append_benchmark_record,
    find_project_root,
)

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "task3_benchmark_results.jsonl"

spark = build_spark_session(app_name="Task3-Partition-Pruning")
register_delta_tables(spark, PROCESSED_DIR)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ca0f9a39-f087-460e-b265-4eb0994536cb;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Registered taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Registered weather: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/weather
Registered air_quality: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/air_quality
Registered taxi_zones: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_zones
Registered integrated_taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips


26/09/17 20:51:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Query pair

Both queries count trips per pickup zone for January 2024 on `taxi_trips` alone
(no join).

In [ ]:
# Baseline: filters on pickup_datetime, which is not a partition column.
BASELINE_SQL = """
SELECT
    pickup_location_id,
    COUNT(trip_id) AS trip_count
FROM taxi_trips
WHERE pickup_datetime >= TIMESTAMP '2024-01-01 00:00:00'
  AND pickup_datetime <  TIMESTAMP '2024-02-01 00:00:00'
GROUP BY pickup_location_id
ORDER BY trip_count DESC
"""

# Optimized: filters directly on pickup_year and pickup_month, which are the partition columns.
OPTIMIZED_SQL = """
SELECT
    pickup_location_id,
    COUNT(trip_id) AS trip_count
FROM taxi_trips
WHERE pickup_year = 2024 AND pickup_month = 1
GROUP BY pickup_location_id
ORDER BY trip_count DESC
"""

## Execution time

In [3]:
baseline_timing = timed_run(spark, "zone_demand_jan2024_baseline", BASELINE_SQL, repetitions=5)
optimized_timing = timed_run(spark, "zone_demand_jan2024_pruned", OPTIMIZED_SQL, repetitions=5)

print("Baseline (filter on pickup_datetime):", baseline_timing)
print("Optimized (filter on pickup_year/pickup_month):", optimized_timing)

Baseline (filter on pickup_datetime): {'query_name': 'zone_demand_jan2024_baseline', 'row_count': 260, 'repetitions': 5, 'all_times_seconds': [0.3725, 0.233, 0.2257, 0.226, 0.22], 'median_seconds': 0.226, 'min_seconds': 0.22, 'max_seconds': 0.3725}
Optimized (filter on pickup_year/pickup_month): {'query_name': 'zone_demand_jan2024_pruned', 'row_count': 260, 'repetitions': 5, 'all_times_seconds': [0.1944, 0.1801, 0.1828, 0.1748, 0.1712], 'median_seconds': 0.1801, 'min_seconds': 0.1712, 'max_seconds': 0.1944}


## Physical plan comparison

The relevant field is `PartitionFilters` on the `Scan parquet` node: it is
absent when Spark cannot statically prune partitions from the query's filter,
and populated with the partition-column predicates when it can. A `Filter`
node appearing immediately after the scan in the baseline plan, and absent
in the optimized plan, is the corresponding structural signature — filtering
that cannot happen at the file-listing stage instead happens row-by-row
after the (unpruned) data has been read.

In [4]:
baseline_plan = capture_explain(spark.sql(BASELINE_SQL), mode="formatted")
optimized_plan = capture_explain(spark.sql(OPTIMIZED_SQL), mode="formatted")

print("=" * 30, "BASELINE PLAN", "=" * 30)
print(baseline_plan)

============================== BASELINE PLAN ==============================
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [pickup_datetime#1, pickup_location_id#7, trip_id#23, pickup_year#27, pickup_month#28]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips]
PushedFilters: [IsNotNull(pickup_datetime), GreaterThanOrEqual(pickup_datetime,2024-01-01 06:00:00.0), LessThan(pickup_datetime,2024-02-01 06:00:00.0)]
ReadSchema: struct<pickup_datetime:timestamp,pickup_location_id:int,trip_id:string>

(2) Filter
Input [5]: [pickup_datetime#1, pickup_location_id#7, trip_id#23, pickup_year#27, pickup_month#28]
Condition

In [5]:
print("=" * 30, "OPTIMIZED PLAN", "=" * 30)
print(optimized_plan)

============================== OPTIMIZED PLAN ==============================
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [pickup_location_id#7, trip_id#23, pickup_year#27, pickup_month#28]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips]
PartitionFilters: [isnotnull(pickup_year#27), isnotnull(pickup_month#28), (pickup_year#27 = 2024), (pickup_month#28 = 1)]
ReadSchema: struct<pickup_location_id:int,trip_id:string>

(2) Project
Output [2]: [pickup_location_id#7, trip_id#23]
Input [4]: [pickup_location_id#7, trip_id#23, pickup_year#27, pickup_month#28]

(3) HashAggregate
Input [2]: [pickup_location_id#7, trip_id#23]
Keys [1]: [pickup_locati

## Correctness verification

The two query variants are checked for row-level equivalence using a
bidirectional set difference: `rows_only_in_before` and `rows_only_in_after`
must both be zero for the result sets to be considered identical (matching
row counts alone would not rule out differing content).

In [6]:
correctness = verify_same_results(spark, BASELINE_SQL, OPTIMIZED_SQL)
print(correctness)
assert correctness["correctness_verified"], "Baseline and optimized queries disagree — investigate before recording a result."


{'correctness_verified': True, 'rows_only_in_before': 0, 'rows_only_in_after': 0}


## Benchmark record

In [7]:
speedup = round(baseline_timing["median_seconds"] / optimized_timing["median_seconds"], 2)

record = {
    "query_name": "zone_demand_jan2024_single_table",
    "technique": "partition_pruning",
    "before_time_sec": baseline_timing["median_seconds"],
    "after_time_sec": optimized_timing["median_seconds"],
    "speedup": speedup,
    "plan_change_summary": (
        "Filtering on pickup_year/pickup_month (the partition columns) instead of "
        "pickup_datetime populates the Scan node's PartitionFilters clause and removes "
        "the row-level Filter node present in the baseline plan: only the "
        "pickup_year=2024/pickup_month=1 partition is read, versus all partitions being "
        "listed and scanned in the baseline."
    ),
    "correctness_verified": correctness["correctness_verified"],
}

append_benchmark_record(record, RESULTS_PATH)
print(f"Speedup: {speedup}x — appended to {RESULTS_PATH}")

Speedup: 1.25x — appended to /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/artifacts/task3_benchmark_results.jsonl


## Discussion

We used a simplified query instead of original Query 1 in analytical_queries.ipynb, because none of the six queries in the library filter by month. Without a month filter, there is nothing for partition pruning to skip.

Our first test included the taxi_zones join and was run without restarting the kernel first. That run showed 1.0x, essentially no speedup. After restarting the kernel and running the exact same query again, we got 1.19x, close to the 1.25x we measured on the simplified version. So the 1.0x result came from leftover state in the session, not from the join itself. Because of this, every timing result in this study was measured in a fresh kernel.

Final result: 1.25x speedup. The physical plan confirms this is real: PartitionFilters shows up and the Filter step disappears once we filter on pickup_year/pickup_month instead of pickup_datetime. The speedup itself is small mainly because the dataset is still small, so reading a few extra partitions doesn't cost much either way.

In [8]:
spark.stop()